In [ ]:
# MWE for running MonteCarlo for different Hamiltonian models

import time
from scipy.linalg import eigh
import numpy as np

from pyscf import gto, scf
from openfermion import hamiltonians
from qarp.operators.compat import from_openfermion

from qarp.operators import JordanWigner
from qarp.operators.pyscf import fermion_operator_from_mf, onv_from_mf

from qarp.blocks import MappedONVStateBlock, TrotterAnsatzBlock, CompositeBlock
from qarp.algorithms import VQE
from qarp.algorithms import TermwiseHadamardTest, StateVector
from qarp.optimizers import ScipyOptimizer
from qarp.engines import QarpEngine

from qarp.algorithms import MonteCarlo, WalkerState

%load_ext autoreload
%autoreload 2
np.set_printoptions(linewidth=1000, suppress=True)


import matplotlib.pyplot as plt
def plot_family(t, trajs, mean_traj, color, label, linestyle=None):
    n = trajs.shape[0]
    for i in range(n):
        plt.plot(t, trajs[i], color=color, alpha=0.2)  # light shade for individuals
    if linestyle is not None:
        plt.plot(t, mean_traj, color="black", linewidth=2.5, label=label, linestyle=linestyle)  # bold mean
    else:
        plt.plot(t, mean_traj, color="black", linewidth=2.5, label=label)  # bold mean


# Hydrogen


In [ ]:
from qarp.blocks import UCCBlock, MappedONVStateBlock, CompositeBlock

bl = 0.735
geometry = f"H 0 0 0; H 0 0 {bl}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True)
mol.build()
mf = scf.RHF(mol)
mf.kernel()

fop = fermion_operator_from_mf(mf)
onv = onv_from_mf(mf)

qham = JordanWigner().encode_operator(fop)
ucc = UCCBlock(onv, generalised=True)
ref = MappedONVStateBlock(onv)
print(onv)
qop_h2 = qham

ham_mat_h2 = qop_h2.sparse_matrix().toarray()

eigenvalues_h2, eigenvectors_h2 = eigh(ham_mat_h2)
h2_gs = eigenvectors_h2[:, np.argmin(eigenvalues_h2)]
h2_gse = eigenvalues_h2[np.argmin(eigenvalues_h2)]
print("Lowest Evals", eigenvalues_h2[:10])

In [ ]:
initial_state_h2 = np.array([1] * (2) + [0] * (2))

In [ ]:
from qarp.operators import VUMPO, qubit_operator_to_mpo

L = 4
n_layers = 2
n_sweeps = 10          # a cap: the sweep stops on `tol` (paper step iv)
maxiter_local = 5      # paper step ii, "locally minimize"
maxiter_global = 1
alpha = 0
hwp = True
mode = "gs"
opt = "local"

# Term-by-term MPO, never dense: quimb site n is qubit n, so no bit reversal.
H_mpo = qubit_operator_to_mpo(qop_h2, L)

print(" ===== MPO Representation of Hamiltonian =====")
print(H_mpo)
print(f"Initial_state:\n {initial_state_h2}\n")

vumpo_h2 = VUMPO(
    H_mpo=H_mpo,
    initial_state=initial_state_h2,
    n_layers=n_layers,
    n_sweeps=n_sweeps,
    maxiter_local=maxiter_local,
    maxiter_global=maxiter_global,
    alpha=alpha,
    hwp=hwp,
    mode=mode,
    opt=opt,
    verbose=False,
)
t0 = time.perf_counter()
params_h2 = vumpo_h2.build(valid_diag=True)
print(f"build: {time.perf_counter() - t0:.2f} s, {vumpo_h2.sweeps_run} sweeps")


In [ ]:
if mode == "gs":
    initial_state = initial_state_h2
else:
    # gs_idx indexes the site-ordered dense U^dag H U: decode to one bit per site, zero-padded
    initial_state = [int(b) for b in np.unravel_index(vumpo_h2.gs_idx, (2,) * L)]
print(vumpo_h2.gs_idx)
print(initial_state_h2)
final_energy = vumpo_h2.compute_energy_tn(params_h2, initial_state=initial_state_h2)
print(f"\n==== Final VUMPO found GS energy ==== {final_energy:.3f}")
print(f"========== Exact GS energy ========== {h2_gse:.3f}")

In [ ]:
from qarp.blocks import VUMPOBrickworkBlock

vumpo_blocks_h2 = VUMPOBrickworkBlock(
    vumpo=vumpo_h2, initial_state=initial_state, optimized_params=params_h2
)

In [ ]:
from qarp.blocks import CompositeBlock

vumpo_block_h2 = CompositeBlock([*vumpo_blocks_h2.blocks], n_qubits=L).build()
vumpo_block_no_basis_h2 = CompositeBlock([*vumpo_blocks_h2.blocks[1:]], n_qubits=L).build()

vumpo_block_h2.build().plot()

vumpo_block_no_basis_h2.build().plot()

In [ ]:
from qarp.blocks import SynthesizedUnitaryBlock
from qarp.algorithms import generate_states_new_basis

U_vumpo_h2 = vumpo_block_h2.build()

walker_states_h2, walkers_circ_h2, _ = generate_states_new_basis(U_vumpo_h2)

walker_states_lab_h2 = [
    WalkerState(state_data=j, sign=1, label=str(i)) for i, j in enumerate(walker_states_h2)
]

walker_circ_lab_h2 = [
    WalkerState(state_data=j, sign=1, label=str(i)) for i, j in enumerate(walkers_circ_h2)
]

engy = np.array(
    [
        np.real(walker_states_lab_h2[l][0].conj().T @ ham_mat_h2 @ walker_states_lab_h2[l][0])
        for l in range(2**L)
    ]
)

print(f"\nTrue Energies\n {eigenvalues_h2}")
print(f"\nUnsorted Energies (VUMPO):\n {engy}")

print(f"\nSorted Energies\n", np.array(sorted(engy)))
reference_energies_sorted_vumpo_h2 = np.array(sorted(engy))
reference_state_index_vumpo_h2 = np.array([np.where(engy == e)[0][0] for e in sorted(engy)])

print(f"\nVUMPO State indices\n {reference_state_index_vumpo_h2}")

In [ ]:
T = 4
delta_tau = 0.01
N0 = 100
csi = 0.1
threshold = 500
samples = 30

# Initialize the simulator
simulator_vumpo_h2 = MonteCarlo(
    hamiltonian=qop_h2,
    # approx_ground_state_energy=reference_energies_sorted_vumpo[0],  # generally the vqe energy
    approx_ground_state_energy=reference_energies_sorted_vumpo_h2[0],
    total_time=T,
    time_step=delta_tau,
    initial_walker_count=N0,
    shift_damping=csi,
    reference_walker_label=int(reference_state_index_vumpo_h2[0]),
    # reference_walker_index=0,
    unitary_block=vumpo_block_h2,
    walker_basis=walker_states_lab_h2,
    population_threshold=threshold,
    num_trajectories=samples,
    mode="Semiclassical",  # "Quantum" or "Semiclassical"
    primitive=StateVector(),  # e.g., "Statevector" or "TermwiseHadamardTest"
    save_walker_history=True,
    history_save_interval=-1,  # -1 to saves last step only
    verbose=True,
)

# BUILD - Build the simulator with U and walker states
simulator_vumpo_h2.build()

# Run the simulation
final_energy_vumpo_h2 = simulator_vumpo_h2.run()

trajectories_smc_vumpo_h2 = simulator_vumpo_h2.energy_estimates_trajectories

print("GS_energy:", np.real(final_energy_vumpo_h2))

trajectories_smc_vumpo_h2 = np.array(trajectories_smc_vumpo_h2)
trajectories_smc_vumpo_h2 = np.array(trajectories_smc_vumpo_h2[:, 0, :])

In [ ]:
import matplotlib.pyplot as plt

t = np.linspace(0, T, int(T / delta_tau))

plt.figure(figsize=(7, 4))

tr_mean_smc_vumpo_h2 = trajectories_smc_vumpo_h2.mean(axis=0)


plot_family(t, trajectories_smc_vumpo_h2, tr_mean_smc_vumpo_h2, color="b", label="vumpo")

plt.hlines(eigenvalues_h2[0], xmin=0, xmax=T, color="black", label=f"Real GS")

# Fermi Hubbard

## Full Diagonalisation

In [ ]:
nx = 3
ny = 2

FH = hamiltonians.fermi_hubbard(
    nx,
    ny,
    -1.0,
    2.5,
    chemical_potential=0.0,
    magnetic_field=1.5,
    periodic=True,
    spinless=True,
    particle_hole_symmetry=False,
)


FH = from_openfermion(FH)  # native boundary
fh_ham = JordanWigner().encode_operator(FH)  # JordanWigner encoding for the Hamiltonian
qop_fh = fh_ham

ham_mat_fh = qop_fh.sparse_matrix().toarray()

eigenvalues_fh, eigenvectors_fh = eigh(ham_mat_fh)
fh_gs = eigenvectors_fh[:, np.argmin(eigenvalues_fh)]
fh_gse = eigenvalues_fh[np.argmin(eigenvalues_fh)]
print("Lowest Evals", eigenvalues_fh[:10])

initial_state_fh = np.array([1] * (2) + [0] * (4))

print(" ===== Fermi Hubbard Initial State =====")
print(initial_state_fh)

In [ ]:
from qarp.operators import VUMPO, qubit_operator_to_mpo

L = 6
n_layers = 8           # deep and narrow: the searched-path regime, see the docs
n_sweeps = 10          # a cap: the sweep stops on `tol`
maxiter_local = 5
maxiter_global = 1
alpha = 0
hwp = True
mode = "diag"  # Target Full diagonalisation
opt = "local"

# Term-by-term MPO, never dense: quimb site n is qubit n, so no bit reversal.
H_mpo = qubit_operator_to_mpo(qop_fh, L)

print(" ===== MPO Representation of Hamiltonian =====")
print(H_mpo)
print(f"Initial_state:\n {initial_state_fh}\n")

vumpo_fh = VUMPO(
    H_mpo=H_mpo,
    initial_state=initial_state_fh,
    n_layers=n_layers,
    n_sweeps=n_sweeps,
    maxiter_local=maxiter_local,
    maxiter_global=maxiter_global,
    alpha=alpha,
    hwp=hwp,
    mode=mode,
    opt=opt,
    verbose=False,
)
t0 = time.perf_counter()
params_fh = vumpo_fh.build(valid_diag=True)
print(f"build: {time.perf_counter() - t0:.2f} s, {vumpo_fh.sweeps_run} sweeps")
# The paper's own quality measures, straight from the MPO (no dense matrix):
print(f"summed energy variance f = {vumpo_fh.energy_variance(params_fh):.4f}")
print(f"off-diagonal ratio       = {vumpo_fh.off_diagonal_ratio(params_fh):.4f}")


In [ ]:
import matplotlib.pyplot as plt

approx_energies_fh = np.array(vumpo_fh.approx_eigs)
exact_energies_fh = np.array(vumpo_fh.exact_eigs)

x = np.arange(len(approx_energies_fh))  # shared x positions
width = 0.35  # width of each bar

plt.bar(x - width / 2, approx_energies_fh, width, label="VUMPO approximate Eigenvalues")
plt.bar(x + width / 2, exact_energies_fh, width, label="Exact eigenvalues")
plt.legend()

In [ ]:
print(f"\n==== Final VUMPO found energies ====\n {approx_energies_fh}\n")
print(f"========== Exact Diagonalization energies ==========\n {exact_energies_fh}")

In [ ]:
from qarp.blocks import VUMPOBrickworkBlock

vumpo_blocks_fh = VUMPOBrickworkBlock(
    vumpo=vumpo_fh, initial_state=initial_state_fh, optimized_params=params_fh
)

In [ ]:
from qarp.blocks import CompositeBlock
vumpo_block_fh = CompositeBlock([*vumpo_blocks_fh.blocks], n_qubits=L).build()
vumpo_block_no_basis_fh = CompositeBlock([*vumpo_blocks_fh.blocks[1:]], n_qubits=L).build()

vumpo_block_fh.build().plot()

vumpo_block_no_basis_fh.build().plot()

In [ ]:
from qarp.blocks import SynthesizedUnitaryBlock
from qarp.algorithms import generate_states_new_basis

U_vumpo_fh = vumpo_block_fh.build()

walker_states_fh, walkers_circ_fh, _ = generate_states_new_basis(U_vumpo_fh)

walker_states_lab_fh = [
    WalkerState(state_data=j, sign=1, label=str(i)) for i, j in enumerate(walker_states_fh)
]

walker_circ_lab_fh = [
    WalkerState(state_data=j, sign=1, label=str(i)) for i, j in enumerate(walkers_circ_fh)
]

engy = np.array(
    [
        np.real(walker_states_lab_fh[l][0].conj().T @ ham_mat_fh @ walker_states_lab_fh[l][0])
        for l in range(2**L)
    ]
)

print(f"\nTrue Energies\n {eigenvalues_fh}")
print(f"\nUnsorted Energies (VUMPO):\n {engy}")

print(f"\nSorted Energies\n", np.array(sorted(engy)))
reference_energies_sorted_vumpo_fh = np.array(sorted(engy))
reference_state_index_vumpo_fh = np.array([np.where(engy == e)[0][0] for e in sorted(engy)])

print(f"\nVUMPO State indices\n {reference_state_index_vumpo_fh}")

In [ ]:
T = 2
delta_tau = 0.01
N0 = 1000
csi = 0.1
threshold = 2000
samples = 10

# Initialize the simulator
simulator_fh = MonteCarlo(
    hamiltonian=qop_fh,
    # approx_ground_state_energy=reference_energies_sorted_vumpo[0],  # generally the vqe energy
    approx_ground_state_energy=reference_energies_sorted_vumpo_fh[0],
    total_time=T,
    time_step=delta_tau,
    initial_walker_count=N0,
    shift_damping=csi,
    reference_walker_label=int(reference_state_index_vumpo_fh[0]),
    # reference_walker_index=0,
    unitary_block=vumpo_block_fh,
    walker_basis=walker_states_lab_fh,
    population_threshold=threshold,
    num_trajectories=samples,
    mode="Semiclassical",  # "Quantum" or "Semiclassical"
    primitive=StateVector(),  # e.g., "Statevector" or "TermwiseHadamardTest"
    save_walker_history=True,
    history_save_interval=-1,  # -1 to saves last step only
    verbose=True,
)

# BUILD - Build the simulator with U and walker states
simulator_fh.build()

# Run the simulation
final_energy_fh = simulator_fh.run()

trajectories_smc_vumpo_fh = simulator_fh.energy_estimates_trajectories

print("GS_energy:", np.real(final_energy_fh))


trajectories_smc_vumpo_fh = np.array(trajectories_smc_vumpo_fh)
trajectories_smc_vumpo_fh = np.array(trajectories_smc_vumpo_fh[:, 0, :])

In [ ]:
import matplotlib.pyplot as plt

t = np.linspace(0, T, int(T / delta_tau))

plt.figure(figsize=(7, 4))

tr_mean_smc_vumpo_fh = trajectories_smc_vumpo_fh.mean(axis=0)

plot_family(t, trajectories_smc_vumpo_fh, tr_mean_smc_vumpo_fh, color="g", label="vumpo")

plt.hlines(eigenvalues_fh[0], xmin=0, xmax=T, color="black", label=f"Real GS")

# Excited State QMC

In [ ]:
num_target_states = 3

approx_excited_energies_vumpo = list(reference_energies_sorted_vumpo_fh[0:num_target_states])
# approx_excited_energies_vumpo = [0.0 for _ in range(num_target_states)]
# approx_gs_energy_vumpo = [0 for _ in range(num_target_states)]
ref_states = [int(i) for i in list(reference_state_index_vumpo_fh[0:num_target_states])]

print(f" ======= Reference States ========:\n {ref_states}")

N0 = [2000] + [2000 for _ in range(num_target_states-1)]
csi = [0.1 for _ in range(num_target_states)]

# Initialize the simulator
T = 1.25
delta_tau = 0.01

threshold = 2000
samples = 10
# Initialize the simulator
simulator_fh_excited = MonteCarlo(
    hamiltonian=qop_fh,
    approx_ground_state_energy=approx_excited_energies_vumpo,  # generally the vqe energy
    total_time=T,
    num_target_states=num_target_states,
    time_step=delta_tau,
    initial_walker_count=N0,
    shift_damping=csi,
    reference_walker_label=ref_states,
    unitary_block=vumpo_block_fh,
    walker_basis=walker_states_lab_fh,
    population_threshold=threshold,
    num_trajectories=samples,
    mode="Semiclassical",  # "Quantum" or "Semiclassical"
    primitive=StateVector(),  # e.g., "Statevector" or "TermwiseHadamardTest"
    save_walker_history=True,
    history_save_interval=-1,  # -1 to saves last step only
    verbose=True,
)

simulator_fh_excited.build()

# Run the simulation
final_energy_fh_excited = simulator_fh_excited.run()

trajectories_smc_vumpo_fh_excited = simulator_fh_excited.energy_estimates_trajectories

trajectories_smc_vumpo_fh_excited = np.array(trajectories_smc_vumpo_fh_excited)

## Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

t = np.linspace(0, T, int(T / delta_tau))

plt.figure(figsize=(7, 4))

colors = ['b', 'g', 'r']
labels = [f"QMC Ground State", f"QMC Excited State 1 (Degenerate)", f"QMC Excited State 2 (Degenerate)"]
linestyles = ['-', '--', '-.']
for j in range(num_target_states):
    trajectories_j = [trajectory[j] for trajectory in trajectories_smc_vumpo_fh_excited]

    trajectories_j = np.array(trajectories_j)
    tr_mean_smc = np.mean(trajectories_j, axis=0)

    t = np.linspace(0, T, int(T / delta_tau))

    plot_family(t, 
        trajectories_j, tr_mean_smc, label=labels[j], color=colors[j], linestyle=linestyles[j]
    )

for j in range(len(eigenvalues_fh[:3])):
    plt.hlines(eigenvalues_fh[j], xmin=0, xmax=T, color=colors[j], label=f"Real {j}th state")


plt.xlabel("T")
plt.ylabel("Energy")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()